# GingivaGen 2.0 — Multi-Material Bio-Scaffold Pipeline

**End-to-end pipeline** converting intraoral `.obj` scans into implantable hybrid gingival scaffolds.

### Triple-Threat Biological Strategy
1. **Mechanical Defense** — 0.5 mm PCL + 10% Bioactive Glass (BAG) armor shell
2. **Bio-Grafting** — Anisotropic Schoen Gyroid GelMA core (325 µm pores)
3. **Vascular/Immune Support** — Sacrificial Pluronic F-127 channels

### Validation Criteria
| Metric | Target | Rationale |
|--------|--------|-----------|
| Mean pore size | 325 ± 25 µm | Fibroblast migration optimum |
| Scaffold stiffness | < 15 KPa | Avoid YAP/TAZ-driven fibrosis |
| Armor thickness | ≥ 0.5 mm | Suture retention |

---
## 1. Install Dependencies & Setup

In [ ]:
%%time
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trimesh', 'porespy', 'fullcontrol', 'scikit-image', 'pyyaml', 'kaggle'],
               check=True, capture_output=True)
print('Dependencies installed')

In [ ]:
import os, sys, shutil
from pathlib import Path

WORK_DIR = '/kaggle/working'
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

# ── Pipeline settings ────────────────────────────────────────────
VOXEL_SIZE_MM = 0.10   # 0.05=production, 0.10=fast, 0.15=fastest
MAX_SCANS     = 5      # batch-processing limit
OUTPUT_DIR    = f'{WORK_DIR}/output'

# ── Locate datasets (mounted via UI, or download via API) ───────
CODE_MOUNTED  = Path('/kaggle/input/gingivagen-2-code')
TEETH_MOUNTED = Path('/kaggle/input/teeth3ds-plus-sample')

if CODE_MOUNTED.exists():
    CODE_DIR = CODE_MOUNTED
else:
    print('Code dataset not mounted — downloading via API...')
    os.makedirs(f'{WORK_DIR}/_datasets', exist_ok=True)
    os.system('kaggle datasets download -d jacqlev/gingivagen-2-code '
              f'-p {WORK_DIR}/_datasets/code --unzip --quiet')
    CODE_DIR = Path(f'{WORK_DIR}/_datasets/code')

if TEETH_MOUNTED.exists():
    TEETH_DIR = TEETH_MOUNTED
else:
    print('Teeth dataset not mounted — downloading via API...')
    os.system('kaggle datasets download -d jacqlev/teeth3ds-plus-sample '
              f'-p {WORK_DIR}/_datasets/teeth --unzip --quiet')
    TEETH_DIR = Path(f'{WORK_DIR}/_datasets/teeth')

print(f'Code dir:  {CODE_DIR}  (exists={CODE_DIR.exists()})')
print(f'Teeth dir: {TEETH_DIR} (exists={TEETH_DIR.exists()})')

In [ ]:
# Copy pipeline code to working dir
for f in CODE_DIR.glob('*.py'):
    shutil.copy2(f, WORK_DIR)
for f in CODE_DIR.glob('*.yaml'):
    shutil.copy2(f, WORK_DIR)

# Clone & patch LisbonTPMS
if not Path('LisbonTPMStool').exists():
    os.system('git clone --quiet https://github.com/SoftwareImpacts/SIMPAC-2025-26.git LisbonTPMStool')
    mf = Path('LisbonTPMStool/mesh_functions.py')
    if mf.exists():
        src = mf.read_text()
        if 'import pymeshlab' in src and 'try:' not in src.split('import pymeshlab')[0][-20:]:
            src = src.replace('import pymeshlab',
                              'try:\n    import pymeshlab\nexcept ImportError:\n    pymeshlab = None')
            mf.write_text(src)

# Verify imports
from orchestrator import GingivaGenV2
from validation_engine import ScaffoldValidator
from mesh_exporter import export_all_materials
print('All imports OK')

# Catalogue scans
obj_files = sorted(TEETH_DIR.rglob('*.obj')) if TEETH_DIR.exists() else []
print(f'Found {len(obj_files)} .obj scans')
for f in obj_files[:10]:
    print(f'  {f}')

---
## 2. Synthetic Smoke Test
Quick validation with a synthetic recession mesh — no dataset needed.

In [ ]:
%%time
import numpy as np, trimesh, yaml, logging
logging.basicConfig(level=logging.INFO,
                    format='%(name)s | %(levelname)s | %(message)s', force=True)

cfg = {}
cfg_path = Path(WORK_DIR) / 'config.yaml'
if cfg_path.exists():
    cfg = yaml.safe_load(cfg_path.read_text()) or {}

# Build synthetic recession mesh
nx, ny = 40, 30
x = np.linspace(-6, 6, nx)
y = np.linspace(-4, 4, ny)
Xm, Ym = np.meshgrid(x, y, indexing='ij')
Zm = 5.0 - 5.0 * np.exp(-(Xm**2 + Ym**2) / 8.0)
verts = np.column_stack([Xm.ravel(), Ym.ravel(), Zm.ravel()])
faces = []
for i in range(nx - 1):
    for j in range(ny - 1):
        v00 = i*ny+j; v10 = (i+1)*ny+j; v01 = i*ny+(j+1); v11 = (i+1)*ny+(j+1)
        faces.extend([[v00, v10, v11], [v00, v11, v01]])
mesh = trimesh.Trimesh(vertices=verts, faces=np.array(faces))
synth_path = f'{WORK_DIR}/synthetic.obj'
mesh.export(synth_path)

# Run pipeline
pipe = GingivaGenV2(output_dir=f'{OUTPUT_DIR}/synthetic', voxel_size=0.15, config=cfg)
synth_results = pipe.run(synth_path)

mat = synth_results['material_grid']
print('\n' + '='*60)
print('SYNTHETIC TEST RESULTS')
print('='*60)
print(f'  Grid shape: {mat.shape}')
for tag, name in [(1,'PCL+BAG armor'),(2,'GelMA core'),(3,'Bio-glue'),(4,'Pluronic')]:
    c = int((mat==tag).sum())
    if c: print(f'  {name}: {c:,} voxels')
val = synth_results.get('validation')
if val:
    print(f'  Pore size: {val.mean_pore_size_um:.1f} um')
    print(f'  Stiffness: {val.scaffold_stiffness_kpa:.2f} KPa')
    print(f'  Pore target met: {val.pore_target_met}')
print('SYNTHETIC: PASS')

---
## 3. Real Scan — Teeth3DS+ Sample

In [ ]:
%%time
real_results = None

if len(obj_files) == 0:
    print('No real scans available — skipping')
else:
    test_obj = str(obj_files[0])
    print(f'Running pipeline on: {test_obj}')
    print(f'Voxel size: {VOXEL_SIZE_MM} mm')
    print('=' * 60)

    pipe2 = GingivaGenV2(output_dir=f'{OUTPUT_DIR}/real_scan', voxel_size=VOXEL_SIZE_MM, config=cfg)
    real_results = pipe2.run(test_obj)

    mat = real_results['material_grid']
    print('\n' + '=' * 60)
    print('REAL SCAN RESULTS')
    print('=' * 60)
    print(f'  Grid shape: {mat.shape}')
    for tag, name in [(1,'PCL+BAG armor'),(2,'GelMA core'),(3,'Bio-glue'),(4,'Pluronic')]:
        c = int((mat==tag).sum())
        if c: print(f'  {name}: {c:,} voxels')
    val = real_results.get('validation')
    if val:
        print(f'  Pore size: {val.mean_pore_size_um:.1f} um')
        print(f'  Porosity: {val.porosity:.3f}')
        print(f'  Stiffness: {val.scaffold_stiffness_kpa:.2f} KPa')
        print(f'  Pore target met: {val.pore_target_met}')
    print(f'  G-code lines: {real_results.get("gcode", "").count(chr(10)):,}')
    print('REAL SCAN: PASS')

---
## 4. Validation Report

In [ ]:
results = real_results if real_results else synth_results
report = results.get('validation')

if report:
    print('=' * 60)
    print('  SCAFFOLD VALIDATION REPORT')
    print('=' * 60)
    print(f'  Mean pore size     : {report.mean_pore_size_um:.1f} um')
    print(f'  Pore size std      : {report.pore_std_um:.1f} um')
    print(f'  Porosity           : {report.porosity:.3f}')
    print(f'  Relative density   : {report.relative_density:.3f}')
    print(f'  Scaffold stiffness : {report.scaffold_stiffness_kpa:.2f} KPa')
    print(f'  Pore target met    : {"YES" if report.pore_target_met else "NO"}')
    if report.stiffness_warning:
        print(f'\n  WARNING: {report.stiffness_warning}')
    print('=' * 60)

    if abs(report.mean_pore_size_um - 325) <= 25:
        print(f'\n  PASS: Pore size {report.mean_pore_size_um:.1f} um within 325+/-25 um')
    else:
        print(f'\n  NOTE: Pore size {report.mean_pore_size_um:.1f} um outside target.')
        print(f'         Coarse voxel sizes may reduce accuracy. Use 0.05mm for clinical.')

    if report.scaffold_stiffness_kpa <= 15.0:
        print(f'  PASS: Stiffness {report.scaffold_stiffness_kpa:.2f} KPa < 15 KPa (safe for YAP/TAZ)')
    else:
        print(f'  FAIL: Stiffness {report.scaffold_stiffness_kpa:.2f} KPa > 15 KPa (risk of fibrosis)')
else:
    print('No validation report available')

---
## 5. Visualization

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

plot_mat = results['material_grid']
plot_title = 'Real Scan' if real_results else 'Synthetic'

colors = ['#FFFFFF', '#C8C8DC', '#64C864', '#FFB432', '#6496FF']
cmap = ListedColormap(colors)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

# Cross-section slices
nz = plot_mat.shape[2]
slices = np.linspace(0, nz - 1, 6, dtype=int)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(f'{plot_title} — Cross-Sections', fontsize=16, fontweight='bold')
for ax, zi in zip(axes.ravel(), slices):
    ax.imshow(plot_mat[:, :, zi].T, cmap=cmap, norm=norm,
              origin='lower', interpolation='nearest', aspect='equal')
    ax.set_title(f'Z-layer {zi}', fontsize=11)
labels = ['Empty', 'PCL+BAG Armor', 'GelMA Core', 'Bio-Glue', 'Pluronic']
patches = [plt.Rectangle((0,0),1,1, facecolor=c) for c in colors]
fig.legend(patches, labels, loc='lower center', ncol=5, fontsize=11,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
plt.savefig(f'{OUTPUT_DIR}/cross_sections.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_DIR}/cross_sections.png')

In [ ]:
# Material distribution per layer
fig, ax = plt.subplots(figsize=(14, 5))
nz = plot_mat.shape[2]
z_layers = np.arange(nz)
bottom = np.zeros(nz)
mat_info = [(1, 'PCL+BAG', '#C8C8DC'), (2, 'GelMA Core', '#64C864'),
            (3, 'Bio-Glue', '#FFB432'), (4, 'Pluronic', '#6496FF')]

for mat_id, name, color in mat_info:
    counts = np.array([(plot_mat[:, :, z] == mat_id).sum() for z in range(nz)])
    if counts.sum() > 0:
        ax.bar(z_layers, counts, bottom=bottom, label=name, color=color, width=1.0)
        bottom += counts

ax.set_xlabel('Z-layer')
ax.set_ylabel('Voxel Count')
ax.set_title(f'{plot_title} — Material Distribution per Layer')
ax.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/material_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Batch Process Multiple Scans

In [ ]:
%%time
import time, traceback

batch_results = []
batch_dir = Path(OUTPUT_DIR) / 'batch'
batch_dir.mkdir(parents=True, exist_ok=True)

scans_to_process = obj_files[:MAX_SCANS]
print(f'Batch processing {len(scans_to_process)} scans at {VOXEL_SIZE_MM} mm voxels')
print('=' * 70)

for i, obj_path in enumerate(scans_to_process):
    scan_name = obj_path.stem
    scan_out = str(batch_dir / scan_name)
    print(f'\n[{i+1}/{len(scans_to_process)}] {scan_name}')
    t0 = time.perf_counter()

    try:
        p = GingivaGenV2(output_dir=scan_out, voxel_size=VOXEL_SIZE_MM, config=cfg)
        res = p.run(str(obj_path))
        elapsed = time.perf_counter() - t0
        m = res.get('material_grid')
        v = res.get('validation')
        entry = {
            'scan': scan_name, 'status': 'OK', 'time_s': round(elapsed, 1),
            'grid_shape': m.shape if m is not None else None,
            'armor_voxels': int((m == 1).sum()) if m is not None else 0,
            'core_voxels': int((m == 2).sum()) if m is not None else 0,
            'pluronic_voxels': int((m == 4).sum()) if m is not None else 0,
            'mean_pore_um': v.mean_pore_size_um if v else None,
            'porosity': v.porosity if v else None,
            'stiffness_kpa': v.scaffold_stiffness_kpa if v else None,
            'pore_target_met': v.pore_target_met if v else None,
            'gcode_lines': res.get('gcode', '').count('\n'),
        }
        print(f'  OK in {elapsed:.1f}s | armor={entry["armor_voxels"]:,} '
              f'core={entry["core_voxels"]:,} pluronic={entry["pluronic_voxels"]:,}')
    except Exception as e:
        elapsed = time.perf_counter() - t0
        entry = {'scan': scan_name, 'status': f'FAIL: {e}', 'time_s': round(elapsed, 1)}
        print(f'  FAILED in {elapsed:.1f}s: {e}')
        traceback.print_exc()

    batch_results.append(entry)

succeeded = sum(1 for r in batch_results if r['status'] == 'OK')
print(f'\n{"=" * 70}')
print(f'Batch complete: {succeeded}/{len(batch_results)} succeeded')

In [ ]:
# Batch comparison chart
if len(batch_results) > 0:
    import pandas as pd
    df = pd.DataFrame(batch_results)
    display(df)

    ok = df[df['status'] == 'OK']
    if len(ok) > 1:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        if 'mean_pore_um' in ok.columns:
            ax = axes[0]
            ax.bar(ok['scan'], ok['mean_pore_um'], color='#64C864')
            ax.axhline(325, color='red', linestyle='--', label='Target (325 um)')
            ax.axhspan(300, 350, alpha=0.15, color='green', label='Tolerance')
            ax.set_ylabel('Mean Pore Size (um)')
            ax.set_title('Pore Size per Scan')
            ax.legend()
            ax.tick_params(axis='x', rotation=45)

        if 'stiffness_kpa' in ok.columns:
            ax = axes[1]
            ax.bar(ok['scan'], ok['stiffness_kpa'], color='#6496FF')
            ax.axhline(15, color='red', linestyle='--', label='YAP/TAZ limit')
            ax.set_ylabel('Scaffold Stiffness (KPa)')
            ax.set_title('Stiffness per Scan')
            ax.legend()
            ax.tick_params(axis='x', rotation=45)

        ax = axes[2]
        x_pos = np.arange(len(ok))
        bottom = np.zeros(len(ok))
        for col, name, color in [('armor_voxels','Armor','#C8C8DC'),
                                  ('core_voxels','Core','#64C864'),
                                  ('pluronic_voxels','Pluronic','#6496FF')]:
            if col in ok.columns:
                vals = ok[col].fillna(0).values.astype(float)
                ax.bar(x_pos, vals, label=name, color=color, bottom=bottom)
                bottom += vals
        ax.set_xticks(x_pos)
        ax.set_xticklabels(ok['scan'], rotation=45, ha='right')
        ax.set_ylabel('Voxel Count')
        ax.set_title('Material Composition')
        ax.legend()

        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/batch_comparison.png', dpi=150, bbox_inches='tight')
        plt.show()

---
## 7. Output Summary

In [ ]:
output_path = Path(OUTPUT_DIR)
all_files = sorted(output_path.rglob('*'))
files_only = [f for f in all_files if f.is_file()]
total_mb = sum(f.stat().st_size for f in files_only) / 1e6
print(f'Output: {len(files_only)} files ({total_mb:.1f} MB)')
for f in files_only:
    print(f'  {f.relative_to(output_path)}  ({f.stat().st_size / 1e3:.1f} KB)')

# Write compact summary for quick retrieval
lines = ['GingivaGen 2.0 Validation Summary']
for label, res in [('SYNTHETIC', synth_results), ('REAL_SCAN', real_results)]:
    if res is None:
        continue
    v = res.get('validation')
    m = res.get('material_grid')
    lines.append(f'\n=== {label} ===')
    if m is not None:
        lines.append(f'grid_shape: {m.shape}')
        for tag, name in [(1,'armor'),(2,'core'),(3,'bioglue'),(4,'pluronic')]:
            lines.append(f'{name}_voxels: {int((m==tag).sum())}')
    if v:
        lines.append(f'mean_pore_um: {v.mean_pore_size_um:.1f}')
        lines.append(f'pore_std_um: {v.pore_std_um:.1f}')
        lines.append(f'porosity: {v.porosity:.3f}')
        lines.append(f'stiffness_kpa: {v.scaffold_stiffness_kpa:.2f}')
        lines.append(f'pore_target_met: {v.pore_target_met}')
        if v.stiffness_warning:
            lines.append(f'stiffness_warning: {v.stiffness_warning}')
summary = '\n'.join(lines)
Path(f'{OUTPUT_DIR}/VALIDATION_SUMMARY.txt').write_text(summary)
print(summary)
print('\nDONE')

---

**Voxel size tradeoffs:**

| Resolution | Voxel | RAM Usage | Time/Scan | Pore Accuracy |
|------------|-------|-----------|-----------|---------------|
| Fast       | 0.15mm | ~2 GB    | ~10s      | Approximate   |
| Balanced   | 0.10mm | ~4 GB    | ~60s      | Good          |
| Production | 0.05mm | ~16 GB   | ~10min    | Clinical-grade|